# FIGARCH: Memoria Longa em Volatilidade

Neste notebook, exploraremos o modelo **FIGARCH** (Fractionally Integrated GARCH),
proposto por **Baillie, Bollerslev & Mikkelsen (1996)**.

O FIGARCH generaliza o modelo GARCH padrao para capturar **dependencia de longo prazo**
na volatilidade, utilizando o operador de **diferenciacao fracionaria** $(1-L)^d$.

**Conteudo:**
1. Memoria longa vs memoria curta
2. O operador de diferenciacao fracionaria $(1-L)^d$
3. Estimando FIGARCH
4. Interpretando o parametro $d$
5. Previsao de volatilidade com memoria longa
6. Exercicios

**Referencias:**
- Baillie, R.T., Bollerslev, T. & Mikkelsen, H.O. (1996). *Fractionally integrated generalized autoregressive conditional heteroskedasticity*. Journal of Econometrics, 74(1), 3-30.
- Bollerslev, T. (1986). *Generalized autoregressive conditional heteroskedasticity*. Journal of Econometrics, 31(3), 307-327.
- Engle, R.F. (1982). *Autoregressive conditional heteroscedasticity with estimates of the variance of United Kingdom inflation*. Econometrica, 50(4), 987-1007.

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
np.random.seed(42)

## 1. Memoria longa vs memoria curta

Um dos **fatos estilizados** mais importantes em series financeiras e que a funcao de
autocorrelacao (ACF) dos retornos absolutos $|r_t|$ e dos retornos ao quadrado $r_t^2$
decai **hiperbolicamente** (lentamente), e nao exponencialmente como previsto pelo GARCH padrao.

- **Memoria curta** (GARCH): $\text{ACF}(k) \sim c \cdot \rho^k$ — decaimento exponencial
- **Memoria longa** (FIGARCH): $\text{ACF}(k) \sim c \cdot k^{2d-1}$ — decaimento hiperbolico

No grafico log-log, a ACF de um processo com memoria longa aparece como uma **reta**
com inclinacao $2d - 1$.

In [ ]:
# TODO: Plote ACF de |returns| e returns^2 ate lag 200
# Dicas:
# - Carregue os dados: data = pd.read_csv('../data/sp500_returns.csv', parse_dates=['date'], index_col='date')
# - returns = data['returns']
# - Calcule abs_returns = np.abs(returns) e sq_returns = returns**2
# - Use plot_long_memory_acf(sq_returns.values, max_lags=200) para ver o decaimento
# - Observe: a ACF decai lentamente, indicando memoria longa
# - No grafico log-log, verifique se a relacao e aproximadamente linear

## 2. O operador de diferenciacao fracionaria $(1-L)^d$

O operador de diferenciacao fracionaria generaliza a diferenciacao inteira:

$$(1-L)^d = \sum_{k=0}^{\infty} \binom{d}{k} (-L)^k = 1 - \sum_{k=1}^{\infty} \delta_k L^k$$

onde os coeficientes $\delta_k$ sao dados recursivamente por:

$$\delta_1 = d, \quad \delta_k = \delta_{k-1} \cdot \frac{k - 1 - d}{k}, \quad k \geq 2$$

Para $0 < d < 1$, os coeficientes $\delta_k > 0$ e decaem hiperbolicamente:
$\delta_k \sim \frac{d}{\Gamma(1-d)} k^{-(1+d)}$ quando $k \to \infty$.

Isso significa que choques passados continuam tendo influencia, mesmo em lags muito distantes.

In [ ]:
# TODO: Calcule os coeficientes de diferenciacao fracionaria para d=0.4
# Dicas:
# - Use _fractional_coefficients(d=0.4, n_lags=100) para calcular os coeficientes
# - Plote os coeficientes: plt.plot(coeffs)
# - Compare com d=0.1 (quase GARCH) e d=0.9 (quase IGARCH)
# - Faca um grafico log-log para verificar o decaimento hiperbolico
# - Observe: maior d => coeficientes decaem mais lentamente => mais memoria

## 3. Estimando FIGARCH

O modelo **FIGARCH(1, d, 1)** e definido por:

$$\sigma_t^2 = \omega + \left[1 - \beta L - \phi L (1-L)^d\right] \epsilon_t^2 + \beta \sigma_{t-1}^2$$

ou equivalentemente:

$$(1 - \beta L) \sigma_t^2 = \omega + \left[1 - \beta L - \phi L (1-L)^d\right] \epsilon_t^2$$

**Parametros:**
- $\omega > 0$: intercepto
- $\phi$: parametro ARCH (analogo ao $\alpha$ no GARCH)
- $d \in (0, 1)$: parametro de diferenciacao fracionaria
- $\beta$: parametro GARCH

**Casos especiais:**
- $d = 0$: GARCH(1,1) padrao
- $d = 1$: IGARCH(1,1)

In [ ]:
# TODO: Estime FIGARCH(1,d,1) com archbox
# Dicas:
# - Carregue os dados se ainda nao carregou
# - data = pd.read_csv('../data/sp500_returns.csv', parse_dates=['date'], index_col='date')
# - returns = data['returns']
# - Crie o modelo: model_fig = FIGARCH(returns.values)
# - Ajuste: results_fig = model_fig.fit()
# - Exiba: print(results_fig.summary())
# - Observe o valor estimado de d: results_fig.params (ordem: omega, phi, d, beta)

## 4. Interpretando o parametro $d$

O parametro $d$ controla a **memoria** do processo de volatilidade:

| Valor de $d$ | Modelo | Memoria | Decaimento da ACF |
|:---:|:---:|:---:|:---:|
| $d = 0$ | GARCH | Curta | Exponencial |
| $0 < d < 0.5$ | FIGARCH | Longa (estacionario) | Hiperbolico |
| $d = 0.5$ | FIGARCH | Longa (fronteira) | Hiperbolico |
| $0.5 < d < 1$ | FIGARCH | Longa (nao-estacionario) | Hiperbolico |
| $d = 1$ | IGARCH | Unitaria | Nao decai |

Na pratica, valores tipicos de $d$ para indices de acoes estao entre **0.3 e 0.5**,
confirmando que a volatilidade tem memoria longa significativa.

In [ ]:
# TODO: Compare GARCH, FIGARCH e IGARCH estimando d
# Dicas:
# - Estime GARCH(1,1): model_garch = GARCH(returns.values, p=1, q=1); res_garch = model_garch.fit()
# - Estime IGARCH(1,1): model_igarch = IGARCH(returns.values); res_igarch = model_igarch.fit()
# - Compare os AIC: res_garch.aic, results_fig.aic, res_igarch.aic
# - Plote as volatilidades condicionais dos 3 modelos sobrepostas
# - Observe: o FIGARCH deve ter o menor AIC se ha memoria longa nos dados
# - Imprima o d estimado do FIGARCH e interprete

## 5. Previsao de volatilidade com memoria longa

Uma diferenca crucial entre GARCH e FIGARCH esta nas **previsoes de longo prazo**:

- **GARCH**: previsoes convergem **exponencialmente** para a variancia incondicional
- **FIGARCH**: previsoes convergem **hiperbolicamente** (muito mais lentamente)

Isso significa que, apos um choque de volatilidade:
- O GARCH "esquece" rapidamente (meia-vida finita)
- O FIGARCH "lembra" por muito mais tempo

Para horizontes curtos (1-5 dias), ambos os modelos dao previsoes similares.
Para horizontes longos (50-100 dias), as diferencas sao substanciais.

In [ ]:
# TODO: Compare forecasts GARCH vs FIGARCH em horizonte longo (100 dias)
# Dicas:
# - forecast_garch = res_garch.forecast(horizon=100)
# - forecast_fig = results_fig.forecast(horizon=100)
# - Plote forecast_garch['volatility'] e forecast_fig['volatility']
# - Adicione linha horizontal para a volatilidade incondicional do GARCH
# - Observe: GARCH converge rapidamente, FIGARCH converge lentamente
# - Calcule: em que horizonte as previsoes divergem significativamente?

## 6. Exercicios

1. **Sensibilidade ao truncamento**: O FIGARCH usa uma expansao truncada do operador
   fracionario. Estime o modelo com `truncation_lag=500` e `truncation_lag=2000`.
   Os resultados mudam significativamente?

2. **Teste de memoria longa**: Compare os AIC de modelos FIGARCH com diferentes
   restricoes sobre $d$. O modelo irrestrito (d estimado) e melhor que $d=0$ (GARCH)
   ou $d=1$ (IGARCH)?

3. **Implicacoes para risco**: Como a memoria longa afeta o calculo de VaR
   para horizontes longos? Compare o VaR de 10 dias usando GARCH vs FIGARCH.

4. **Dados reais**: Aplique o FIGARCH a retornos de diferentes classes de ativos
   (acoes, cambio, commodities). O parametro $d$ varia entre classes?

In [ ]:
# TODO: Teste diferentes valores de d e compare AIC
# Dicas:
# - Estime FIGARCH com truncation_lag=500: FIGARCH(returns.values, truncation_lag=500)
# - Estime FIGARCH com truncation_lag=2000: FIGARCH(returns.values, truncation_lag=2000)
# - Compare os parametros estimados e AIC
# - Faca uma tabela comparativa:
#   | Modelo | omega | phi | d | beta | AIC |
#   |--------|-------|-----|---|------|-----|
# - Qual modelo tem o melhor ajuste (menor AIC)?